In [63]:
import numpy as np
EPOCHS = 100
LEARNING_RATE = 0.1

class MatchingAlgorithm:
    def __init__(self, n_features, learning_rate=LEARNING_RATE):
        """
        Initializes the algorithm.

        Args:
            n_features (int): The number 'n' of features or functions F to measure.
            learning_rate (float): The learning rate 'η' for weight updates.
        """
        self.n_features = n_features
        self.eta = learning_rate
        self.alphas = np.full(n_features, 1.0 / n_features)

    def _calculate_features(self, a, b):
        """
        Calculates the feature vector F(a,b).

        This is the domain-specific part (the functions F_i).
        For this example, we will use 3 features:
        - F_1: Age similarity
        - F_2: Music interest similarity
        - F_3: Sports interest similarity

        Returns a numpy array [F_1, F_2, F_3].
        """
        # F_1: Age similarity (1.0 = same age, 0.0 = +10 years difference)
        age_diff = abs(a['age'] - b['age'])
        f1_age = max(0.0, 1.0 - (age_diff / 100.0))

        # F_2: Music similarity (1.0 = identical, 0.0 = opposite)
        f2_music = 1.0 - abs(a['interest_music'] - b['interest_music'])

        # F_3: Sports similarity (1.0 = identical, 0.0 = opposite)
        f3_sports = 1.0 - abs(a['interest_sports'] - b['interest_sports'])

        return np.array([f1_age, f2_music, f3_sports])

    def predict(self, a, b):
        """
        Calculates the matching score A(a,b) for a pair of users.
        A(a,b) = Σ(α_i * F_i(a,b))
        """
        # Gets the vector [F_1, F_2, F_3, ...]
        features = self._calculate_features(a, b);

        # Calculates the dot product
        # (α_1*F_1) + (α_2*F_2) + (α_3*F_3)
        return np.dot(self.alphas, features)

    def train(self, training_data, epochs=EPOCHS):
        """
        Updates the 'α' weights using backpropagation and gradient descent.

        Args:
            training_data (list): A list of tuples (user_a, user_b, y_label),
            where y_label is 0 (bad match) or 1 (good match).
            epochs (int): Number of times the training data is processed.
        """
        #print(f"--- Starting training for {epochs} epochs ---")
        for epoch in range(epochs):
            # Initializes the total gradient for this epoch
            # [∂L/∂α_1, ∂L/∂α_2, ..., ∂L/∂α_n]
            gradients = np.zeros(self.n_features)

            total_loss = 0

            # Iterates over all training examples (a, b)
            for a, b, y_true in training_data:

                # 1. Calculate the current prediction A(a,b)
                y_pred = self.predict(a, b)

                # 2. Calculate the prediction error (A(a,b) - y_ab)
                error = y_pred - y_true

                # 3. Calculate the loss (Mean Squared Error) for this pair
                loss = error ** 2
                total_loss += loss

                # 4. Get the feature vector F_k(a,b)
                features = self._calculate_features(a, b)

                # 5. Calculate the gradient contribution for this pair
                # 2 * (A(a,b) - y_ab) * F_k(a,b)
                gradient_contribution = 2 * error * features

                # 6. Add to the total gradient
                gradients += gradient_contribution

            # 7. Update all 'α' weights simultaneously
            # α_k = α_k - η * (gradient_k)
            self.alphas -= self.eta * gradients

            # Print progress (optional)
            if (epoch + 1) % 20 == 0:
              print(f"Epoch {epoch+1}/{epochs}, Total Loss (L): {total_loss:.4f}")

        #print("--- Training finished ---")

    def find_top_matches(self, user_a, user_list, top_x=3):
        """
        Finds the 'top_x' best users for 'user_a' from a list.
        """
        scores = []

        # Calculates the score A(a,b) for each 'b' in the list
        for user_b in user_list:
            if user_a == user_b: # Do not match with self
                continue
            score = self.predict(user_a, user_b)
            scores.append((user_b['name'], score))

        # Sorts the list of scores in descending order
        scores.sort(key=lambda x: x[1], reverse=True)

        # Returns the 'top_x' results
        return scores[:top_x]

In [64]:
a = {
    'name': 'Ana',
    'age': 28,
    'interest_music': 0.9,
    'interest_sports': 0.2
}

candidate_list = [
    { 'name': 'Beto', 'age': 28, 'interest_music': 0.7, 'interest_sports': 0.3 },
    { 'name': 'Carla', 'age': 29, 'interest_music': 0.7, 'interest_sports': 0.9 },
    { 'name': 'David', 'age': 60, 'interest_music': 0.1, 'interest_sports': 0.1 },
    { 'name': 'Elena', 'age': 75, 'interest_music': 0.1, 'interest_sports': 0.4 }
]

# Train the model for a (with their data)
b2_train = { 'age': 90, 'interest_music': 0.7, 'interest_sports': 0.4 }
b3_train = { 'age': 80, 'interest_music': 0.2, 'interest_sports': 0.5 }
b4_train = { 'age': 70, 'interest_music': 0.8, 'interest_sports': 0.5 }
b5_train = { 'age': 18, 'interest_music': 0.7, 'interest_sports': 0.4 }
b6_train = { 'age': 22, 'interest_music': 0.2, 'interest_sports': 0.5 }
b7_train = { 'age': 19, 'interest_music': 0.8, 'interest_sports': 0.5 }

training_data = [(a, b2_train, 1)]

model = MatchingAlgorithm(n_features=3, learning_rate=LEARNING_RATE)
model.train(training_data, epochs=EPOCHS)

print("--- Final Weights (α) ---")
print(f" (α_1: Age, α_2: Music, α_3: Sports)")
print(f" {model.alphas}")
print("\n")

top_matches = model.find_top_matches(a, candidate_list, top_x=3)

for name, score in top_matches:
    print(f"  -> {name} (Score A(a,b): {score:.4f})")

training_data = [(a, b2_train, 1),(a, b3_train, 1)]

model = MatchingAlgorithm(n_features=3, learning_rate=LEARNING_RATE)
model.train(training_data, epochs=EPOCHS)

print("--- Final Weights (α) ---")
print(f" (α_1: Age, α_2: Music, α_3: Sports)")
print(f" {model.alphas}")
print("\n")

top_matches = model.find_top_matches(a, candidate_list, top_x=3)

for name, score in top_matches:
    print(f"  -> {name} (Score A(a,b): {score:.4f})")

training_data = [
    (a, b2_train, 1),
    (a, b3_train, 1),
    (a, b4_train, 1)]

model = MatchingAlgorithm(n_features=3, learning_rate=LEARNING_RATE)
model.train(training_data, epochs=EPOCHS)

print("--- Final Weights (α) ---")
print(f" (α_1: Age, α_2: Music, α_3: Sports)")
print(f" {model.alphas}")
print("\n")

top_matches = model.find_top_matches(a, candidate_list, top_x=3)

for name, score in top_matches:
    print(f"  -> {name} (Score A(a,b): {score:.4f})")

training_data = [
    (a, b2_train, 1),
    (a, b3_train, 1),
    (a, b4_train, 1),
    (a, b5_train, 0)]

model = MatchingAlgorithm(n_features=3, learning_rate=LEARNING_RATE)
model.train(training_data, epochs=EPOCHS)

print("--- Final Weights (α) ---")
print(f" (α_1: Age, α_2: Music, α_3: Sports)")
print(f" {model.alphas}")
print("\n")

top_matches = model.find_top_matches(a, candidate_list, top_x=3)

for name, score in top_matches:
    print(f"  -> {name} (Score A(a,b): {score:.4f})")

training_data = [
    (a, b2_train, 1),
    (a, b3_train, 1),
    (a, b4_train, 1),
    (a, b5_train, 0),
    (a, b6_train, 0)
]

model = MatchingAlgorithm(n_features=3, learning_rate=LEARNING_RATE)
model.train(training_data, epochs=EPOCHS)

print("--- Final Weights (α) ---")
print(f" (α_1: Age, α_2: Music, α_3: Sports)")
print(f" {model.alphas}")
print("\n")

top_matches = model.find_top_matches(a, candidate_list, top_x=3)

for name, score in top_matches:
    print(f"  -> {name} (Score A(a,b): {score:.4f})")

training_data = [
    (a, b2_train, 1),
    (a, b3_train, 1),
    (a, b4_train, 1),
    (a, b5_train, 0),
    (a, b6_train, 0),
    (a, b7_train, 0)
]

model = MatchingAlgorithm(n_features=3, learning_rate=LEARNING_RATE)
model.train(training_data, epochs=EPOCHS)

print("--- Final Weights (α) ---")
print(f" (α_1: Age, α_2: Music, α_3: Sports)")
print(f" {model.alphas}")
print("\n")

top_matches = model.find_top_matches(a, candidate_list, top_x=3)

for name, score in top_matches:
    print(f"  -> {name} (Score A(a,b): {score:.4f})")

Epoch 20/100, Total Loss (L): 0.0000
Epoch 40/100, Total Loss (L): 0.0000
Epoch 60/100, Total Loss (L): 0.0000
Epoch 80/100, Total Loss (L): 0.0000
Epoch 100/100, Total Loss (L): 0.0000
--- Final Weights (α) ---
 (α_1: Age, α_2: Music, α_3: Sports)
 [0.42403819 0.52429093 0.52429093]


  -> Beto (Score A(a,b): 1.3153)
  -> Carla (Score A(a,b): 0.9965)
  -> David (Score A(a,b): 0.8651)
Epoch 20/100, Total Loss (L): 0.0218
Epoch 40/100, Total Loss (L): 0.0107
Epoch 60/100, Total Loss (L): 0.0052
Epoch 80/100, Total Loss (L): 0.0026
Epoch 100/100, Total Loss (L): 0.0013
--- Final Weights (α) ---
 (α_1: Age, α_2: Music, α_3: Sports)
 [0.8064608  0.10047852 0.79231082]


  -> Beto (Score A(a,b): 1.5999)
  -> David (Score A(a,b): 1.2816)
  -> Carla (Score A(a,b): 1.1165)
Epoch 20/100, Total Loss (L): 0.0317
Epoch 40/100, Total Loss (L): 0.0131
Epoch 60/100, Total Loss (L): 0.0055
Epoch 80/100, Total Loss (L): 0.0025
Epoch 100/100, Total Loss (L): 0.0012
--- Final Weights (α) ---
 (α_1: Age, 